# Imports

In [ ]:
import subprocess
import json
from typing import Dict, List, Optional, Any, Union
from dataclasses import dataclass, field, asdict
from pathlib import Path
import sys

# Config

In [ ]:
@dataclass
class PromptConfig:
    """
    Configuration for a single prompt test case.
    
    Attributes:
        prompt: The prompt text to send to the model
        variables: Optional dictionary of variables to interpolate into the prompt
        assert_rules: Optional list of assertion rules to validate the output
    """
    prompt: str
    variables: Optional[Dict[str, str]] = None
    assert_rules: Optional[List[Dict[str, Any]]] = None
    
    def __post_init__(self) -> None:
        """Validate prompt configuration after initialization."""
        if not isinstance(self.prompt, str) or not self.prompt.strip():
            raise ValueError("Prompt must be a non-empty string")
        
        if self.variables is not None and not isinstance(self.variables, dict):
            raise TypeError("Variables must be a dictionary")
        
        if self.assert_rules is not None and not isinstance(self.assert_rules, list):
            raise TypeError("Assert rules must be a list")

# Ollama Provider

In [ ]:
@dataclass
class OllamaProvider:
    """
    Configuration for Ollama model provider.
    
    Attributes:
        model_name: Name of the Ollama model to test (e.g., 'llama2', 'mistral')
        base_url: Base URL for Ollama API (default: http://localhost:11434)
        temperature: Sampling temperature for model responses (0.0 to 1.0)
        max_tokens: Maximum number of tokens in the response
    """
    model_name: str
    base_url: str = "http://localhost:11434"
    temperature: float = 0.7
    max_tokens: int = 1000
    
    def __post_init__(self) -> None:
        """Validate Ollama provider configuration."""
        if not isinstance(self.model_name, str) or not self.model_name.strip():
            raise ValueError("Model name must be a non-empty string")
        
        if not isinstance(self.base_url, str) or not self.base_url.startswith("http"):
            raise ValueError("Base URL must be a valid HTTP/HTTPS URL")
        
        if not 0.0 <= self.temperature <= 2.0:
            raise ValueError("Temperature must be between 0.0 and 2.0")
        
        if not isinstance(self.max_tokens, int) or self.max_tokens <= 0:
            raise ValueError("Max tokens must be a positive integer")
    
    def to_config(self) -> Dict[str, Any]:
        """Convert provider configuration to promptfoo format."""
        return {
            "id": f"ollama:{self.model_name}",
            "config": {
                "apiBaseUrl": self.base_url,
                "temperature": self.temperature,
                "max_tokens": self.max_tokens
            }
        }

In [ ]:
@dataclass
class PromptfooConfig:
    """
    Main configuration for promptfoo testing suite.
    
    Attributes:
        providers: List of model providers to test
        prompts: List of prompt configurations to evaluate
        output_path: Path to save test results (default: promptfoo-results.json)
        default_test: Optional default test configuration
    """
    providers: List[OllamaProvider]
    prompts: List[PromptConfig]
    output_path: Path = field(default_factory=lambda: Path("promptfoo-results.json"))
    default_test: Optional[Dict[str, Any]] = None
    
    def __post_init__(self) -> None:
        """Validate promptfoo configuration."""
        if not self.providers:
            raise ValueError("At least one provider must be specified")
        
        if not self.prompts:
            raise ValueError("At least one prompt must be specified")
        
        if not isinstance(self.output_path, Path):
            self.output_path = Path(self.output_path)
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert configuration to promptfoo-compatible dictionary format."""
        config = {
            "providers": [provider.to_config() for provider in self.providers],
            "prompts": [prompt.prompt for prompt in self.prompts],
        }
        
        # Add test assertions if any prompts have them
        tests = []
        for prompt in self.prompts:
            if prompt.assert_rules:
                test = {"assert": prompt.assert_rules}
                if prompt.variables:
                    test["vars"] = prompt.variables
                tests.append(test)
        
        if tests:
            config["tests"] = tests
        elif self.default_test:
            config["defaultTest"] = self.default_test
        
        return config

# Promptfoo Runner

In [ ]:
class PromptfooRunner:
    """
    Runner class for executing promptfoo CLI commands and managing test execution.
    """
    
    def __init__(self, config: PromptfooConfig) -> None:
        """
        Initialize the promptfoo runner with configuration.
        
        Args:
            config: PromptfooConfig instance containing test configuration
        
        Raises:
            TypeError: If config is not a PromptfooConfig instance
        """
        if not isinstance(config, PromptfooConfig):
            raise TypeError("Config must be a PromptfooConfig instance")
        
        self.config = config
        self.config_file = Path("promptfooconfig.yaml")
        print("Promptfoo runner initialized successfully")
    
    def check_dependencies(self) -> bool:
        """
        Check if required dependencies (promptfoo CLI and Ollama) are installed.
        
        Returns:
            bool: True if all dependencies are available, False otherwise
        """
        try:
            # Check promptfoo CLI
            result = subprocess.run(
                ["promptfoo", "--version"],
                capture_output=True,
                text=True,
                timeout=10
            )
            
            if result.returncode != 0:
                print("Promptfoo CLI is not installed or not in PATH")
                return False
            
            print(f"Promptfoo CLI found: {result.stdout.strip()}")
            
            # Check Ollama availability
            result = subprocess.run(
                ["curl", "-s", f"{self.config.providers[0].base_url}/api/tags"],
                capture_output=True,
                text=True,
                timeout=10
            )
            
            if result.returncode != 0:
                print("Ollama server may not be running. Please ensure Ollama is started.")
                return False
            
            print("Ollama server is accessible")
            return True
            
        except subprocess.TimeoutExpired:
            print("Dependency check timed out")
            return False
        except FileNotFoundError as e:
            print(f"Required command not found: {e}")
            return False
        except Exception as e:
            print(f"Error checking dependencies: {e}")
            return False
    
    def save_config(self) -> None:
        """
        Save the promptfoo configuration to a YAML file.
        
        Raises:
            IOError: If unable to write configuration file
        """
        try:
            import yaml
            
            config_dict = self.config.to_dict()
            
            with open(self.config_file, 'w', encoding='utf-8') as f:
                yaml.dump(config_dict, f, default_flow_style=False, allow_unicode=True)
            
            print(f"Configuration saved to {self.config_file}")
            
        except ImportError:
            print("PyYAML is not installed. Installing via pip...")
            subprocess.run([sys.executable, "-m", "pip", "install", "pyyaml"], check=True)
            self.save_config()  # Retry after installation
            
        except Exception as e:
            print(f"Failed to save configuration: {e}")
            raise IOError(f"Unable to write configuration file: {e}")
    
    def run_evaluation(self) -> Optional[Dict[str, Any]]:
        """
        Execute the promptfoo evaluation using the CLI.
        
        Returns:
            Optional[Dict[str, Any]]: Evaluation results as a dictionary, or None if failed
        """
        try:
            # Save configuration file
            self.save_config()
            
            # Run promptfoo eval command
            print("Starting promptfoo evaluation...")
            result = subprocess.run(
                ["promptfoo", "eval", "-c", str(self.config_file), "-o", str(self.config.output_path)],
                capture_output=True,
                text=True,
                timeout=300  # 5 minutes timeout
            )
            
            if result.returncode != 0:
                print(f"Evaluation failed: {result.stderr}")
                return None
            
            print("Evaluation completed successfully")
            print(f"Output: {result.stdout}")
            
            # Load and return results
            return self.load_results()
            
        except subprocess.TimeoutExpired:
            print("Evaluation timed out after 5 minutes")
            return None
        except Exception as e:
            print(f"Error during evaluation: {e}")
            return None
    
    def load_results(self) -> Optional[Dict[str, Any]]:
        """
        Load evaluation results from the output file.
        
        Returns:
            Optional[Dict[str, Any]]: Results dictionary, or None if file not found
        """
        try:
            if not self.config.output_path.exists():
                print(f"Results file not found: {self.config.output_path}")
                return None
            
            with open(self.config.output_path, 'r', encoding='utf-8') as f:
                results = json.load(f)
            
            print(f"Results loaded from {self.config.output_path}")
            return results
            
        except json.JSONDecodeError as e:
            print(f"Invalid JSON in results file: {e}")
            return None
        except Exception as e:
            print(f"Error loading results: {e}")
            return None
    
    def cleanup(self) -> None:
        """Remove temporary configuration files."""
        try:
            if self.config_file.exists():
                self.config_file.unlink()
                print("Temporary configuration file removed")
        except Exception as e:
            print(f"Failed to cleanup configuration file: {e}")

# Results

In [ ]:
# Define test prompts with assertions
prompts = [
    PromptConfig(
        prompt="Explain quantum computing in simple terms.",
        assert_rules=[
            {"type": "contains", "value": "quantum"},
            {"type": "contains", "value": "computing"}
        ]
    ),
    PromptConfig(
        prompt="Write a haiku about {{topic}}.",
        variables={"topic": "artificial intelligence"},
        assert_rules=[
            {"type": "contains", "value": "haiku"},
            {"type": "javascript", "value": "output.split('\\n').length >= 3"}
        ]
    ),
    PromptConfig(
        prompt="What is the capital of France? Answer in one word.",
        assert_rules=[
            {"type": "icontains", "value": "Paris"}
        ]
    )
]

In [ ]:
# Configure Ollama provider
providers = [
    OllamaProvider(
        model_name="llama3.2:3b",  # Change to your installed model
        temperature=0.7,
        max_tokens=500
    )
]

In [ ]:
# Create promptfoo configuration
config = PromptfooConfig(
    providers=providers,
    prompts=prompts,
    output_path=Path("evaluation_results.json")
)

In [ ]:
# Initialize and run evaluation
runner = PromptfooRunner(config)

In [ ]:
# Check dependencies before running
if not runner.check_dependencies():
    print("Required dependencies are not available. Exiting.")

In [ ]:
# Execute evaluation
results = runner.run_evaluation()

In [ ]:
if results:
    print("=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(json.dumps(results, indent=2))
else:
    print("Evaluation failed or produced no results")

In [ ]:
# Cleanup temporary files
runner.cleanup()

In [ ]:
!promptfoo view